### Preliminary analysis of the GRI

Using standard models for survival analysis with competing risks such Cause specific Cox PH and Fine and Grey. 

We will run it over two datasets, one would be the subset of qpis (quality measures for colorectal cancer), 
and secondly over the whole clinical data, where there are potential predictors. The dataset has been preprocessed in "prpeprocess-1.ipynb", where expert knowledge from Joanne Edwards' lab was considered to substract the relevant columns and rows, plus any further decisions taken. 

Hopefully the second dataset would allow give us hints of future qpis. 

Regarding missing values, they need to be imputed... the continuous values standarized, etc. 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import math

# sksurv
from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis

# scikit-learn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer

# cmprsk (contains Fine and Grey model) it is and python 
# wrapper of R functions. Dependencies such r-survival are also downloaded
# Requires rpy2 librar
import rpy2

# my functions 
import helper
from helper import fit_cs_cox_manual
from helper import cifs_from_cs_hazards
from helper import fit_preprocessor
import metrics
from metrics.score import CompRiskMetricsCompute

# Iterative inputer (for mice)
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold 
from sklearn.ensemble import ExtraTreesRegressor

# lifelines
from lifelines import CoxPHFitter

pd.set_option("display.max_columns", None)



### 5-fold cross-validation


We can use MICE for imputation

In [ ]:
## prepare the data

gri = pd.read_csv("data/GRI_preprocess-1.csv", index_col="TMA_order")
print(gri.shape)
# define time and status
time, status = "DFS_months_2020", "Cmprsk_Status"

print(gri[[time, status]].isna().sum())

# drop if we do not know any of these two:
df = gri.dropna(subset=[time, status])

print(df.shape)
# create surv object 
#y = Surv.from_dataframe(time, status, df) # does not work because it expects binary
#y = Surv.from_dataframe(time, event = status > 0) 

X = df.drop(columns=[time, status, "Unnamed: 0"])
T_all = df[time].astype(float).to_numpy()
D_all = df[status].astype(int).to_numpy()



In [ ]:
X.head()

In [ ]:
X.dtypes
X.info()

We can use the pipe to introduce preprocessing with mice and scaling. 

In [ ]:
def modelling_pipe(model):
    # select the numeric
    num_sel = make_column_selector(dtype_include=np.number)
    # select the categorical
    cat_sel = make_column_selector(dtype_include=["object", "category", "string", "bool"])

    num = Pipeline([
        # Using the equivalent to Mice in R
        ("mice", IterativeImputer(random_state=42, sample_posterior=True, max_iter=10, initial_strategy="median")),
        ("scaler", StandardScaler()),
        ])
    cat = Pipeline([
        # since it is categorical we impute with the most frequent
        ("mode", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ])
    prep = ColumnTransformer([
        ("num", num, num_sel),
        ("cat", cat, cat_sel)],
        remainder="drop")
    
    return Pipeline([("prep", prep),("model", model)])
    

We can use a stratified 5cv fold so that we preserve the proportions of causes by fold. 

In [ ]:
# init object cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(cv.get_n_splits())  


In [ ]:
# generate the splits based on the causes D
splits = list(cv.split(np.arange(len(df)), y=D_all))
len(splits[0][0]) 


Each fold having similar proportions of causes, as well as train and sets

In [ ]:
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"Fold number: {fold}")
    print(f"Length of training set: {len(train_idx)}")
    print(f"Length of test set: {len(val_idx)}")

    vals, counts = np.unique(D_all[train_idx], return_counts=True)
    pct = counts / counts.sum() * 100

    print(f"Training set proportion of each cause: {pct}")

    vals, counts = np.unique(D_all[val_idx], return_counts=True)
    pct = counts / counts.sum() * 100

    print(f"Test set proportion of each cause: {pct}")


Train a Fine and Grey model, and a cause specific cox proportional hazards. 

- The cause-specific cox proportional hazards fits a separate cox PH to each cause, and therefore it treats the events from other causes as censored. We know that it would overestimate the incidence of the events. 
- The Fine and Grey model, or subdistribution hazards, where the cumulative incidence is associated with the subdistribution hazard. Here the competing risk events are treated differently, by a weigth estimated as if there had informative censoring. It also assumes a PH form. 

https://pmc.ncbi.nlm.nih.gov/articles/PMC5326634/#:~:text=Cause%2Dspecific%20hazard%20regression%20model%20can%20be%20fit%20with%20Cox,with%20COX%20proportional%20hazard%20regression.

We could use the pipe, but the problem is that the majority of models might not come from sklearn, and wrappers might be constantly needed. 

In [ ]:
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    # subsetting
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    t_train, t_val = T_all[train_idx], T_all[val_idx]
    d_train, d_val = D_all[train_idx], D_all[val_idx]

    #print(np.max(t_train))
    
    # time grid for evaluation purposes
    time_grid = np.arange(0, np.max(t_train), 0.1)
    #print(len(time_grid))

    cause_1 = (d_train == 1)
    pipe_trm = modelling_pipe(CoxPHSurvivalAnalysis(alpha=1.0))
    pipe_trm.fit(X_train, Surv.from_arrays(event=cause_1, time=t_train))
    ## something like this
    ## calculate cumhaz on the time grid... 


Trying with manual helper function that do something similar to the pipes


In [ ]:

ibs_c1_folds = []
ibs_c2_folds = []

def chf_to_matrix(chf_list, tgrid):
    n = len(chf_list)
    m = len(tgrid)
    out = np.empty((n, m), float)
    for i, f in enumerate(chf_list):
        out[i] = f(tgrid)        # evaluate cumulative hazard at all times
    return out

for fold, (train_idx, val_idx) in enumerate(splits, 1):
    # 1) subsetting
    print("Fold:", fold)
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    t_train, t_val = T_all[train_idx], T_all[val_idx]
    d_train, d_val = D_all[train_idx], D_all[val_idx]

    #print(np.max(t_train))
    
    # 2) time grid for evaluation purposes
    time_grid = np.arange(0, np.max(t_train), 0.1)
    
    # 3) inpute and scale on TRAIN
    print("Processing with impute/scaling/one_hot encoding on TRAIN:")
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()    
    cat_cols = X_train.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

    #mice gives some warning, probably because small sample size vs covariates
    #mice = IterativeImputer(random_state=42, sample_posterior=True,
    #                        max_iter=10, initial_strategy="median")

    mice = IterativeImputer(
        random_state=42,
        max_iter=20,           
        tol=1e-3, 
        sample_posterior=False,
        initial_strategy="median",
    )
    # maybe try the ExtraTreesRegressor or the KNN ?
    Xn_imp = mice.fit_transform(X_train[num_cols]) if num_cols else np.empty((len(X_train), 0))

    scaler = StandardScaler()
    Xn = scaler.fit_transform(Xn_imp) if num_cols else np.empty((len(X_train), 0))

    cat_imp = SimpleImputer(strategy="most_frequent")
    Xc_imp = cat_imp.fit_transform(X_train[cat_cols]) if cat_cols else np.empty((len(X_train), 0))

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", drop=None, sparse=False)
    Xc = ohe.fit_transform(Xc_imp) if cat_cols else np.empty((len(X_train), 0))

    print("Xn:", type(Xn), getattr(Xn, "shape", None), "ndim=", getattr(Xn, "ndim", None))
    print("Xc:", type(Xc), getattr(Xc, "shape", None), "ndim=", getattr(Xc, "ndim", None))

    # --- combine & remove constant columns
    X_comb = np.hstack([Xn, Xc]) if (Xn.size or Xc.size) else np.empty((len(X_train), 0))
    nzv = VarianceThreshold(threshold=0.0)
    Xnz = nzv.fit_transform(X_comb) if X_comb.size else X_comb

    print("Xnz:", type(Xnz), getattr(Xnz, "shape", None), "ndim=", getattr(Xnz, "ndim", None))
    
    # 4) inpute and scale on TEST
    print("Processing with impute/scaling/one_hot encoding on TEST:")
    num_cols = X_val.select_dtypes(include=[np.number]).columns.tolist()    
    cat_cols = X_val.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

    # numeric
    if num_cols:
        Xn_val_imp = mice.transform(X_val[num_cols])
        Xn_val = scaler.transform(Xn_val_imp)
    else:
        Xn_val = np.empty((len(X_val), 0))

    # categorical
    if cat_cols:
        Xc_val_imp = cat_imp.transform(X_val[cat_cols])
        Xc_val = ohe.transform(Xc_val_imp)  # dense because sparse_output=False
    else:
        Xc_val = np.empty((len(X_val), 0))

    print("Xn:", type(Xn_val), getattr(Xn_val, "shape", None), "ndim=", getattr(Xn_val, "ndim", None))
    print("Xc:", type(Xc_val), getattr(Xc_val, "shape", None), "ndim=", getattr(Xc_val, "ndim", None))

    # combine and apply same NZV mask
    X_comb_val = np.hstack([Xn_val, Xc_val]) if (Xn_val.size or Xc_val.size) else np.empty((len(X_val), 0))
    Xnz_val = nzv.transform(X_comb_val) if X_comb_val.size else X_comb_val

    print("Xnz:", type(Xnz_val), getattr(Xnz_val, "shape", None), "ndim=", getattr(Xnz_val, "ndim", None))

    # 5) Fit model 

    y_c1 = Surv.from_arrays(event=(d_train == 1), time=t_train)
    y_c2 = Surv.from_arrays(event=(d_train == 2), time=t_train)

    # adding alpha as regularizer for ridge regression penalty
    cox_1 = CoxPHSurvivalAnalysis(alpha=1).fit(Xnz, y_c1)
    cox_2 = CoxPHSurvivalAnalysis(alpha=1).fit(Xnz, y_c2)

    # 6) Estimate the cumulative hazard
    m = len(time_grid)
    A1 = cox_1.predict_cumulative_hazard_function(Xnz_val)
    A2 = cox_2.predict_cumulative_hazard_function(Xnz_val)
    # make it as matrix
    A1_mat = chf_to_matrix(A1, time_grid)   # shape (n_val, m)
    A2_mat = chf_to_matrix(A2, time_grid)

    # 7) Estimate the CIF form the cumulative hazard
    F1_pred, F2_pred = cifs_from_cs_hazards(A1_mat, A2_mat)

    # 8) Get the brier score with comprks and adjustment for censoring
    metrics = CompRiskMetricsCompute(
        time_grid=time_grid,
        durations_train=t_train,
        delta_train=d_train,
        eps=1e-6
    )

    ibs_c1 = metrics.integrated_brier_cif(F1_pred, t_val, d_val, cause=1)
    ibs_c2 = metrics.integrated_brier_cif(F2_pred, t_val, d_val, cause=2)

    ibs_c1_folds.append(float(ibs_c1))
    ibs_c2_folds.append(float(ibs_c2))
    print(f"Fold {fold}: IBS(cause1)={ibs_c1:.4f} | IBS(cause2)={ibs_c2:.4f}")






    

Somewhat works.... but it needs to be fixed. Something is not working properly.